# Predicting whether an order should be sent to a technical approver

## Part 1: Load and examine the data

In [108]:
import os
from dotenv import load_dotenv

load_dotenv()

data_bucket = os.getenv("DATA_BUCKET")
subfolder = os.getenv("SUBFOLDER")
dataset = os.getenv("DATASET")
role = os.getenv("ROLE")
region = os.getenv("REGION")

model_name = os.getenv("MODEL_NAME")
endpoint_config_name = os.getenv("ENDPOINT_CONFIG_NAME")

In [109]:
import pandas as pd
from time import sleep

import boto3
import sagemaker
from sagemaker.train.configs import InputData
from sklearn.model_selection import train_test_split

#role = sagemaker.get_execution_role()
s3 = boto3.client("s3", region_name=region)

In [110]:
df = pd.read_csv(s3.get_object(Bucket=data_bucket, Key=f"{subfolder}/incoming/{dataset}")["Body"])
df.head()

,tech_approval_required,requester_id,role,product,quantity,price,total
0,0,E2300,tech,Desk,1,664,664
1,0,E2300,tech,Keyboard,9,649,5841
2,0,E2374,non-tech,Keyboard,1,821,821
3,1,E2374,non-tech,Desktop Computer,24,655,15720
4,0,E2327,non-tech,Desk,1,758,758


In [85]:
print(f'Number of rows in dataset: {df.shape[0]}')
print(df[df.columns[0]].value_counts())

Number of rows in dataset: 299
tech_approval_required
0    240
1     59
Name: count, dtype: int64


## Part 2: Get the data into the right shape

In [86]:
encoded_data = pd.get_dummies(df)
encoded_data.head()

,tech_approval_required,quantity,price,total,requester_id_E2300,requester_id_E2301,requester_id_E2303,requester_id_E2304,requester_id_E2307,requester_id_E2309,...,requester_id_E2399,requester_id_E2400,role_non-tech,role_tech,product_Chair,product_Cleaning,product_Desk,product_Desktop Computer,product_Keyboard,product_Mouse
0,0,1,664,664,True,False,False,False,False,False,...,False,False,False,True,False,False,True,False,False,False
1,0,9,649,5841,True,False,False,False,False,False,...,False,False,False,True,False,False,False,False,True,False
2,0,1,821,821,False,False,False,False,False,False,...,False,False,True,False,False,False,False,False,True,False
3,1,24,655,15720,False,False,False,False,False,False,...,False,False,True,False,False,False,False,True,False,False
4,0,1,758,758,False,False,False,False,False,False,...,False,False,True,False,False,False,True,False,False,False


In [87]:
corrs = encoded_data.corr()['tech_approval_required'].abs()
columns = corrs[corrs > .1].index
corrs = corrs.filter(columns)
corrs

tech_approval_required      1.000000
quantity                    0.375480
total                       0.272536
requester_id_E2322          0.108833
requester_id_E2324          0.116835
requester_id_E2367          0.116835
requester_id_E2368          0.116835
requester_id_E2386          0.161707
role_non-tech               0.113948
role_tech                   0.113948
product_Chair               0.146474
product_Cleaning            0.186325
product_Desk                0.307350
product_Desktop Computer    0.969390
product_Keyboard            0.219507
product_Mouse               0.205957
Name: tech_approval_required, dtype: float64

In [88]:
encoded_data = encoded_data[columns]
encoded_data.head()

,tech_approval_required,quantity,total,requester_id_E2322,requester_id_E2324,requester_id_E2367,requester_id_E2368,requester_id_E2386,role_non-tech,role_tech,product_Chair,product_Cleaning,product_Desk,product_Desktop Computer,product_Keyboard,product_Mouse
0,0,1,664,False,False,False,False,False,False,True,False,False,True,False,False,False
1,0,9,5841,False,False,False,False,False,False,True,False,False,False,False,True,False
2,0,1,821,False,False,False,False,False,True,False,False,False,False,False,True,False
3,1,24,15720,False,False,False,False,False,True,False,False,False,False,True,False,False
4,0,1,758,False,False,False,False,False,True,False,False,False,True,False,False,False


## Part 3: Create training, validation and test data sets

In [89]:
from sagemaker.train.configs import InputData
train_df, val_and_test_data = train_test_split(encoded_data, test_size=0.3, random_state=0)
val_df, test_df = train_test_split(val_and_test_data, test_size=0.333, random_state=0)

s3.put_object(Bucket=data_bucket, Key=f"{subfolder}/processed/train.csv", Body=train_df.to_csv(index=False, header=False))
s3.put_object(Bucket=data_bucket, Key=f"{subfolder}/processed/val.csv", Body=val_df.to_csv(index=False, header=False))
s3.put_object(Bucket=data_bucket, Key=f"{subfolder}/processed/test.csv", Body=test_df.to_csv(index=False, header=True))

train_input = InputData(channel_name="train",
                        data_source=f"s3://{data_bucket}/{subfolder}/processed/train.csv",
                        content_type="text/csv")

val_input = InputData(channel_name="validation",
                      data_source=f"s3://{data_bucket}/{subfolder}/processed/val.csv",
                      content_type="text/csv") 

## Part 4: Train the model

In [90]:
from sagemaker.train.configs import InputData, Compute, OutputDataConfig, StoppingCondition
from sagemaker.core import image_uris
from sagemaker.train import ModelTrainer

# XGBoost container
region = boto3.Session().region_name

container = image_uris.retrieve(framework="xgboost", region=region, version="1.7-1", instance_type="ml.m4.xlarge")

# Compute
compute = Compute(instance_type="ml.m4.xlarge", instance_count=1, enable_managed_spot_training=True)

# Trainer
stopping_condition = StoppingCondition(max_runtime_in_seconds=1200, max_wait_time_in_seconds=2000)
model_trainer = ModelTrainer(training_image=container,
                             role=role,
                             compute=compute,
                             stopping_condition=stopping_condition,
                             output_data_config=OutputDataConfig(s3_output_path=f"s3://{data_bucket}/{subfolder}/output"),
                             hyperparameters={"max_depth": "5",
                                              "subsample": "0.7",
                                              "objective": "binary:logistic",
                                              "eval_metric": "auc",
                                              "num_round": "70",
                                              "early_stopping_rounds": "10"})

# Start training
model_trainer.train(input_data_config=[train_input, val_input])

/home/pasar/.venvs/sagemaker/lib/python3.14/site-packages/rich/live.py:260: UserWarning: install "ipywidgets" for 
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

In [91]:
sm = boto3.client("sagemaker", region_name=region)

# Find the most recent training job with this prefix
response = sm.list_training_jobs(SortBy="CreationTime", SortOrder="Descending", MaxResults=100)

matching_jobs = [
    job for job in response["TrainingJobSummaries"]
    if job["TrainingJobName"].startswith(model_trainer.base_job_name)
]

if not matching_jobs:
    raise RuntimeError("No matching SageMaker training jobs found.")

training_job_name = matching_jobs[0]["TrainingJobName"]

# Retrieve the training results
job = sm.describe_training_job(TrainingJobName=training_job_name)
model_artifact = job.get("ModelArtifacts", {}).get("S3ModelArtifacts")

print("Job:", training_job_name)
print("Status:", job["TrainingJobStatus"])
print("Training time:", job.get("TrainingTimeInSeconds"), "seconds")
print("Billable time:", job.get("BillableTimeInSeconds"), "seconds")

print("\nFinal metrics:")
for metric in job.get("FinalMetricDataList", []):
    print(f'{metric["MetricName"]}: {metric["Value"]}')

print("\nModel artifact:")
print(model_artifact)

Job: sagemaker-xgboost-job-20261010134702
Status: Completed
Training time: 129 seconds
Billable time: 50 seconds

Final metrics:
validation:auc: 1.0
train:auc: 0.9987900257110596

Model artifact:
s3://test-bucket-psardinha/decision-making/output/sagemaker-xgboost-job-20261010134702/output/model.tar.gz


# Part 5: Test the model
Evaluate the model with test dataset, but without exposing it to the internet

In [96]:

from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
import numpy as np
import uuid

region = boto3.Session().region_name or "eu-central-1"
sm = boto3.client("sagemaker", region_name=region)

try:
    sm.delete_model(ModelName=model_name)
except Exception as e:
    pass

sm.create_model(
    ModelName=model_name,
    ExecutionRoleArn=role,
    PrimaryContainer={
        "Image": container,
        "ModelDataUrl": model_artifact,
    },
)
print(f"Created evaluation model: {model_name}")

test_df = pd.read_csv(f's3://{data_bucket}/{subfolder}/processed/test.csv')
y_true = test_df.iloc[:, 0].astype(int).to_numpy() 
# Keep existing logic: features start at column 1.
features_df = test_df.iloc[:, 1:].copy()
# Convert Boolean values to numeric values.
for column in features_df.columns:
    features_df[column] = features_df[column].map(lambda value: int(value) if isinstance(value, (bool, np.bool_)) else value)
eval_key = f"{subfolder}/evaluation/eval.csv"
eval_file = f"s3://{data_bucket}/{eval_key}"
s3.put_object(Bucket=data_bucket, Key=eval_key, Body=features_df.to_csv(index=False, header=False))    

# Delete evaluation output file if it exists
eval_output_key = f"{subfolder}/evaluation/eval.csv.out"
eval_output_file = f"{data_bucket}/{eval_output_key}"
s3.delete_object(Bucket=data_bucket, Key=eval_output_key)

output_uri = f"s3://{data_bucket}/{subfolder}/evaluation"
transform_job_name = f"order-approval-evaluation-{uuid.uuid4().hex[:8]}"
sm.create_transform_job(TransformJobName=transform_job_name,
                        ModelName=model_name,
                        TransformInput={"DataSource": {"S3DataSource": {"S3DataType": "S3Prefix",
                                                                        "S3Uri": eval_file}},
                                        "ContentType": "text/csv",
                                        "SplitType": "Line"},
                        TransformOutput={
                            "S3OutputPath": output_uri,
                            "Accept": "text/csv",
                            "AssembleWith": "Line"},
                        TransformResources={
                            "InstanceType": "ml.m5.large",
                            "InstanceCount": 1})
print(f"Started transform job: {transform_job_name}")

waiter = sm.get_waiter("transform_job_completed_or_stopped")
waiter.wait(TransformJobName=transform_job_name,  WaiterConfig={"Delay": 20, "MaxAttempts": 180})
job = sm.describe_transform_job(TransformJobName=transform_job_name)
if job["TransformJobStatus"] != "Completed":
    raise RuntimeError(f"Batch Transform failed: {job.get('FailureReason')}")
print(f"Transform job: {transform_job_name} completed successfully.")

# XGBoost binary:logistic returns probability scores.
df = pd.read_csv(s3.get_object(Bucket=data_bucket, Key=eval_output_key)["Body"], header=None)
s3.delete_object(Bucket=data_bucket, Key=eval_output_key)
scores = df.iloc[:, 0].astype(float).to_numpy()    
if len(scores) != len(y_true):
    raise RuntimeError(f"Label/prediction count mismatch: {len(y_true)} labels and {len(scores)} predictions")
y_pred = (scores >= 0.5).astype(int)

auc = roc_auc_score(y_true, scores)
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)

print("\nEvaluation metrics")
print(f"ROC AUC:   {auc:.4f}")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")

# Remove the temporary SageMaker model resource.
sm.delete_model(ModelName=model_name)

Created evaluation model: order-approval-model
Started transform job: order-approval-evaluation-1374be1b
Transform job: order-approval-evaluation-1374be1b completed successfully.

Evaluation metrics
ROC AUC:   0.5000
Accuracy:  0.8000
Precision: 0.0000
Recall:    0.0000


{'ResponseMetadata': {'RequestId': '2b192def-8b78-4533-b0f1-875a84afd1ae',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '2b192def-8b78-4533-b0f1-875a84afd1ae',
   'strict-transport-security': 'max-age=47304000; includeSubDomains',
   'x-frame-options': 'DENY',
   'content-security-policy': "frame-ancestors 'none'",
   'cache-control': 'no-cache, no-store, must-revalidate',
   'x-content-type-options': 'nosniff',
   'content-type': 'application/x-amz-json-1.1',
   'date': 'Sat, 10 Oct 2026 13:04:03 GMT',
   'content-length': '0'},
  'RetryAttempts': 0}}

## Part 6: Host the model

In [97]:
import time

endpoint_name = 'order-approval'
sm = boto3.client("sagemaker", region_name="eu-central-1")
try:
    sm.delete_endpoint(EndpointName=endpoint_name)
    print("Existing endpoint deleted.")
    time.sleep(30)
except sm.exceptions.ClientError as e:
    if "Could not find endpoint" in str(e):
        pass
    else:
        raise

In [98]:
import boto3

region = boto3.Session().region_name or "eu-central-1"
sm = boto3.client("sagemaker", region_name=region)

try:
    sm.delete_model(ModelName=model_name)
    print("Existing model deleted.")
    time.sleep(10)
except sm.exceptions.ClientError as e:
    if f'Could not find model "{model_name}"' in str(e):
        pass
    else:
        raise

response = sm.create_model(
    ModelName=model_name,
    ExecutionRoleArn=role,
    PrimaryContainer={
        "Image": container,
        "ModelDataUrl": model_artifact,
    },
)

print("Created model:", response["ModelArn"])

Created model: arn:aws:sagemaker:eu-central-1:801796542461:model/order-approval-model


In [99]:
try:
    sm.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
    print("Existing endpoint configuration deleted.")
    time.sleep(10)
except sm.exceptions.ClientError as e:
    if f'Could not find endpoint configuration "{endpoint_config_name}"' in str(e):
        pass
    else:
        raise

endpoint_name = "order-approval"

sm.create_endpoint_config(EndpointConfigName=endpoint_config_name,
                          ProductionVariants=[{"VariantName": "AllTraffic",
                                              "ModelName": model_name,
                                              "InitialInstanceCount": 1,
                                              "InstanceType": "ml.m4.xlarge",
                                              "InitialVariantWeight": 1.0}])

response = sm.create_endpoint(EndpointName=endpoint_name, EndpointConfigName=endpoint_config_name)
print("Endpoint creation requested:", endpoint_name)

Endpoint creation requested: order-approval


## Part 7: (Re)Test the model, this time in Production
This is a repetition of previous section 5, but now with the endpoint exposed to the internet

In [105]:
import time

while True:
    response = sm.describe_endpoint(
        EndpointName=endpoint_name
    )
    status = response["EndpointStatus"]
    print("Endpoint status:", status)

    if status in ("InService", "Failed"):
        break

    time.sleep(30)

if status == "Failed":
    print(response.get("FailureReason", "No failure reason returned"))

Endpoint status: InService


In [101]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

runtime = boto3.client("sagemaker-runtime", region_name="eu-central-1")

def get_prediction(row):
    features = row.iloc[1:]  # Exclude label in column 0

    # Convert Boolean features to numeric values
    features = features.map(lambda value: int(value) if isinstance(value, bool) else value)
    
    payload = ",".join(map(str, features.tolist()))

    response = runtime.invoke_endpoint(
        EndpointName=endpoint_name,
        ContentType="text/csv",
        Body=payload.encode("utf-8"),
    )

    score = float(response["Body"].read().decode("utf-8").strip())
    return int(score >= 0.5)

cols = list(test_df.columns)
test_df["prediction"] = test_df.apply(get_prediction, axis=1)
test_df = test_df[["prediction"] + cols]
test_df.head(10)

# Actual labels from the test dataset
y_true = test_df.iloc[:, 1].astype(int)
# Predicted labels from your SageMaker endpoint
y_pred = test_df.iloc[:, 0].astype(int)
# Calculate metrics
print("Accuracy:", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred, zero_division=0))
print("Recall:", recall_score(y_true, y_pred, zero_division=0))
print("F1-score:", f1_score(y_true, y_pred, zero_division=0))
print("\nConfusion matrix:")
print(confusion_matrix(y_true, y_pred))
print("\nClassification report:")
print(classification_report(y_true, y_pred, zero_division=0))

Accuracy: 0.8
Precision: 0.0
Recall: 0.0
F1-score: 0.0

Confusion matrix:
[[24  0]
 [ 6  0]]

Classification report:
              precision    recall  f1-score   support

           0       0.80      1.00      0.89        24
           1       0.00      0.00      0.00         6

    accuracy                           0.80        30
   macro avg       0.40      0.50      0.44        30
weighted avg       0.64      0.80      0.71        30



## Part 8: Remove the Endpoint (optional)
Removes the endpoint exposed to the internet

In [107]:
sm = boto3.client("sagemaker", region_name="eu-central-1")
try:
    sm.delete_endpoint(EndpointName=endpoint_name)
    print("Existing endpoint deleted.")
    time.sleep(30)
except sm.exceptions.ClientError as e:
    if "Could not find endpoint" in str(e):
        pass
    else:
        raise

try:
    sm.delete_model(ModelName=model_name)
    print("Existing model deleted.")
    time.sleep(10)
except sm.exceptions.ClientError as e:
    if f'Could not find model "{model_name}"' in str(e):
        pass
    else:
        raise
    
try:
    sm.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
    print("Existing endpoint configuration deleted.")
except sm.exceptions.ClientError as e:
    if f'Could not find endpoint configuration "{endpoint_config_name}"' in str(e):
        pass
    else:
        raise 